# Match the Dish to Its Rizz — full solution notebook

End-to-end run of the `src/` pipeline on Kaggle GPUs. The metric is
**R-precision**, and on this data R-precision equals the image -> category
accuracy under the balanced assignment, so everything below is about making a
101-way food classifier as accurate as possible.

**Before you run:** turn the accelerator on (T4 x2 or P100 are enough) and
internet on for the one-time backbone download.

Edit `BACKBONE` first:

| BACKBONE | speed on T4 | quality |
| --- | --- | --- |
| `siglip2-b16-384` | ~10 min | good baseline |
| `siglip2-so400m-384` | ~40 min | strong, recommended |
| `dfn5b-h14-378` | ~2 h | strongest per image, use with `VIEWS = ["center", "full"]` |
| `pe-bigG-448` | ~6 h, 2xT4 | best zero-shot; only if you have the hours |

Then run the cells in order. Embeddings are cached in `features/`, so re-running
after a config change is cheap.


In [ ]:
# --- configuration ---------------------------------------------------------
BACKBONE = "siglip2-so400m-384"          # see the table above
VIEWS = ["center", "full", "flip_center", "flip_full"]
BATCH_SIZE = 32
NUM_WORKERS = 4
RUN_LORA = False                          # only if zero-shot + probe is not enough
LORA_LR = 1e-4
LORA_EPOCHS = 6
ADD_VALIDATION_LABELS = True              # +1010 labelled images for the final model
USE_PROBE = True                          # 5-fold prompt-initialised linear head
SINKHORN_TEMP = 0.35

!pip install -q open_clip_torch timm transformers pandas numpy scipy scikit-learn pillow tqdm
import sys, pathlib, importlib

REPO = "/kaggle/input/kaggle-assignment"          # <- point at your uploaded repo
DATA = None                                        # None = auto-detect /kaggle/input/*/train.csv
sys.path.insert(0, f"{REPO}/assignment_3/src")

for m in ["config", "catalog", "dataset", "features", "prompts", "balance",
          "textmap", "imagemap", "retrieval", "metrics", "pipeline", "evaluate"]:
    importlib.import_module(m)

import config, evaluate as ev, imagemap, pipeline
print("backbone:", BACKBONE)

In [ ]:
# --- locate the data and encode everything (cached) -------------------------

def build_cfg(**over):
    image_cfg = imagemap.ImageMapConfig(
        weight_zero_shot=0.3, weight_linear=0.4, weight_knn=0.0,
        prop_alpha=0.0, balance="sinkhorn", sinkhorn_temp=SINKHORN_TEMP,
    )
    cfg = pipeline.PipelineConfig(
        backbones=(BACKBONE,), views=tuple(VIEWS), batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, precision="auto", use_cache=True,
        add_validation_labels=False, image_cfg=image_cfg,
    )
    for k, v in over.items():
        setattr(cfg, k, v)
    return cfg

root = config.resolve_data_dir(DATA)
print("data dir:", root)

## 1. Validate

R-precision here should already be far above the 0.5 starter baseline. If
`queries -> right category` is anything but 1.0, dump the decisions and fix the
text side first - a single wrong category costs that query its whole score.

If `images -> right category` is the bottleneck (it always is), move to the next
cells: probe, then LoRA, then an ensemble of two backbones.


In [ ]:
import catalog, metrics

ground_truth = catalog.load_ground_truth(root)
result = pipeline.run_pipeline(root, "validation", build_cfg(), ground_truth=ground_truth)
scores = metrics.evaluate_rankings(result.rankings, catalog.relevant_map(ground_truth))
print(f"R-precision {scores['r_precision']:.4f} | mAP@20 {scores['mAP@20']:.4f} | "
      f"P@5 {scores['P@5']:.4f} | Hit@1 {scores['Hit@1']:.4f} | MRR@20 {scores['MRR@20']:.4f}")
print("query category accuracy:", result.diagnostics.get("query_category_accuracy"))
for name, acc in result.diagnostics.get("image_category_accuracy", {}).items():
    print(f"  {name:<12} top1 {acc['top1']:.4f} top5 {acc['top5']:.4f}")

## 2. Prompt-initialised 5-fold probe (cheap, usually +1-2 points)

Trains a linear head on the frozen embeddings, initialised from the text
prototypes, and ensembles the five fold heads.


In [ ]:
import subprocess, sys as _sys

if USE_PROBE:
    !python3 {REPO}/assignment_3/src/train.py --data {root} --mode probe \
        --backbone {BACKBONE} --views {" ".join(VIEWS)} --folds 5 --epochs 20 \
        --batch-size {BATCH_SIZE} --out {REPO}/assignment_3/models

heads = sorted(str(p) for p in pathlib.Path(f"{REPO}/assignment_3/models/probe").glob("probe_fold*.npz"))
print("heads:", heads)

In [ ]:
probe_cfg = build_cfg()
if USE_PROBE:
    probe_cfg.image_cfg.head_paths = tuple(heads)

result = pipeline.run_pipeline(root, "validation", probe_cfg, ground_truth=ground_truth)
scores = metrics.evaluate_rankings(result.rankings, catalog.relevant_map(ground_truth))
print(f"with probe heads: R-precision {scores['r_precision']:.4f}")
for name, acc in result.diagnostics.get("image_category_accuracy", {}).items():
    print(f"  {name:<12} top1 {acc['top1']:.4f}")

## 3. LoRA fine-tuning (only if the probe is clearly stuck)

Low-rank adapters on the vision tower, with the text tower frozen and the
classifier weights initialised from the prompt ensemble. Checkpoints on best
validation image top-1.


In [ ]:
if RUN_LORA:
    !python3 {REPO}/assignment_3/src/train.py --data {root} --mode lora \
        --backbone {BACKBONE} --views {" ".join(VIEWS)} --epochs {LORA_EPOCHS} \
        --lr {LORA_LR} --batch-size {BATCH_SIZE} --lora-rank 16 --lora-alpha 32 \
        --out {REPO}/assignment_3/models
    # reload the adapter before encoding: features.load_model picks it up via
    # train.load_lora when you pass --lora-checkpoint (see README).

## 4. Ensemble two backbones

Independent pretraining corpora disagree exactly where the confusable categories
live (pasta, cakes, rice, noodle soups, beef cuts). Concatenating the two
embedding spaces simply sums their cosine votes - `backbones=(A, B)`.


In [ ]:
# ENSEMBLE = ("siglip2-so400m-384", "dfn5b-h14-378")
ENSEMBLE = None
if ENSEMBLE:
    ens_cfg = build_cfg(backbones=ENSEMBLE)
    result = pipeline.run_pipeline(root, "validation", ens_cfg, ground_truth=ground_truth)
    scores = metrics.evaluate_rankings(result.rankings, catalog.relevant_map(ground_truth))
    print(f"ensemble {ENSEMBLE}: R-precision {scores['r_precision']:.4f}")

## 5. Final submission

Re-uses the winning configuration and additionally trains the image-side head on
the labelled validation gallery (+1010 images, +50% supervision). Validate the
file with the organisers' own checker before uploading.


In [ ]:
import retrieval

best = probe_cfg if USE_PROBE else build_cfg()
best.backbones = ENSEMBLE or (BACKBONE,)
best.add_validation_labels = ADD_VALIDATION_LABELS

result = pipeline.run_pipeline(root, "test", best)
retrieval.write_submission(result.rankings, result.queries["query_id"].tolist(),
                           "submission.csv", top_k=best.top_k)
print("wrote submission.csv")
!python3 {root}/validate_submission.py submission.csv --data-dir {root}